# Chronos-Bolt versus frozen seasonal baselines

Eight series across Tetouan, Portugal, a French household and Victoria. Forecasts:
24 and 168 hours, with the exact test origins from the completed baseline pilot.
No fine-tuning or test-based model selection. The model receives the last 336 hourly
observations ending at each origin. Missing contexts are not filled.

**Before running:** Runtime > Change runtime type > T4 GPU. Then run cells in order.
Upload `chronos_pilot_inputs.zip` when prompted. This bundle contains the local
experiment inputs and code, so no GitHub push or repository clone is needed.

The model is Chronos-Bolt base, chosen to continue the existing project. It is not
the newest Chronos model. The 0.5 quantile is used because MAE is the primary metric.
Bolt has a native 64-step horizon; the 168-hour task uses its built-in long-horizon
extension, which can degrade accuracy. There is no guarantee these public datasets
were absent from pretraining. This notebook does not establish statistical significance.

References: [official code](https://github.com/amazon-science/chronos-forecasting),
[model configuration](https://huggingface.co/amazon/chronos-bolt-base/blob/main/config.json).

In [ ]:
%pip install -q "chronos-forecasting==2.2.2" "transformers>=4.48,<5" "numpy>=1.26,<3" "pandas>=2.2,<3"

In [ ]:
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime, then rerun."
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

## Upload the frozen inputs

Choose the supplied input ZIP. Its checksum and every extracted input will be
checked. This uploads public electricity data and experiment files to your Colab
runtime. No credentials or private project files are included.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, zipfile

ROOT = Path("/content/chronos_pilot")
ROOT.mkdir(exist_ok=True)
uploaded = files.upload()
assert len(uploaded) == 1, "Select only chronos_pilot_inputs.zip"
filename, payload = next(iter(uploaded.items()))
assert hashlib.sha256(payload).hexdigest() == "7a2d448b10e49821159fcdc55e47c42939be7684eccc3755442ed271d80f15db", "Wrong or changed bundle"
with zipfile.ZipFile(filename) as archive:
    for member in archive.infolist():
        parts = PurePosixPath(member.filename)
        assert not parts.is_absolute() and ".." not in parts.parts and "\\" not in member.filename
        assert (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
del uploaded, payload
print("Inputs ready:", ROOT)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "src/chronos_pilot.py"),
                "--root", str(ROOT), "--check-only"], check=True)

## Run the forecasts

The first run downloads model weights. Progress is printed per batch. Completed
series/horizon files are reused if you rerun with the same configuration.
Progress in `/content` survives rerunning cells, but not deletion of the runtime.
If interrupted, use the download cell to save completed files before disconnecting.

The default batch size is 8 because the 168-step extension increases GPU memory
use. If CUDA runs out of memory, restart the runtime and use batch size 2 with a
new output directory. Do not change model/context after looking at test scores.

In [ ]:
OUTPUT = ROOT / "reports/chronos_pilot"
subprocess.run([sys.executable, "-u", str(ROOT / "src/chronos_pilot.py"),
                "--root", str(ROOT), "--output", str(OUTPUT), "--batch-size", "8"], check=True)

In [ ]:
import pandas as pd
table = pd.read_csv(OUTPUT / "comparison.csv")
display(table.loc[table.baseline == "selected", ["dataset", "series", "horizon_hours",
    "baseline_rule", "chronos_MAE", "baseline_MAE", "chronos_MAE_gain_pct"]])
print("Positive gain means lower MAE for Chronos. These are descriptive results.")

## Download results

Download the ZIP, retain it locally, and return it to the workspace for analysis.
It includes every prediction, actuals, origins, baseline predictions, model commit,
package versions and comparison scores. `complete.json` indicates all 16 cases
finished. Without that marker, results are partial.

Next analysis will compare paired errors while accounting for time dependence.
Trained LightGBM/XGBoost models are a separate later experiment.

In [ ]:
import shutil
assert OUTPUT.exists(), "No results yet"
archive_path = shutil.make_archive("/content/chronos_pilot_results", "zip", OUTPUT)
files.download(archive_path)